In [17]:
import pandas as pd

df = pd.read_csv('mismatch_summary.tsv', sep='\t')
df2 = pd.read_csv('matched_summary.tsv', sep='\t')

read from Normal_1_2_3.mgf and create a dictionary with scan number as key and dataset string as value: BEGIN IONS
PEPMASS=490.28207
CHARGE=2
CHARGE_ID=2
MSLEVEL=2
COLLISION_ENERGY=0.0
FILENAME=
SEQ=LLIYGGSTR
PROTEIN=
SCANS=1
SCAN=1
PROVENANCE_FILENAME=MSV000086484/ccms_peak/Raw_Spectrum_Files/Prostate_Urine_01_20_25Jun11_Earth_11-04-23.mzML
PROVENANCE_SCAN=8432, here key is 1 and dataset is MSV000086484

In [18]:
mgf_file = 'Normal_1_2_3.mgf'
scan_to_dataset = {}

with open(mgf_file, 'r') as file:
    dataset = None
    scan_number = None
    for line in file:
        line = line.strip()
        if line.startswith("BEGIN IONS"):
            dataset = None
            scan_number = None
        elif line.startswith("PROVENANCE_FILENAME="):
            provenance = line.split('=')[1]
            if provenance.startswith('ProteomeCentral'):
                parts = provenance.split('/')
                dataset = parts[1] if len(parts) > 1 else None
            else:
                dataset = provenance.split('/')[0]
        elif line.startswith("SCAN="):
            scan_number = int(line.split('=')[1])
        elif line.startswith("END IONS") and scan_number is not None and dataset is not None:
            scan_to_dataset[scan_number] = dataset

print(scan_to_dataset)

{1: 'MSV000086484', 2: 'PXD004423', 3: 'PXD004423', 4: 'PXD004423', 5: 'PXD004423', 6: 'PXD004423', 7: 'PXD004423', 8: 'PXD004423', 9: 'PXD004423', 10: 'PXD004423', 11: 'PXD003935', 12: 'PXD033989', 13: 'MSV000086484', 14: 'PXD020011', 15: 'PXD020011', 16: 'PXD020011', 17: 'PXD020011', 18: 'PXD020011', 19: 'PXD020011', 20: 'PXD020011', 21: 'PXD020011', 22: 'PXD020011', 23: 'PXD020011', 24: 'PXD020011', 25: 'PXD020011', 26: 'PXD020011', 27: 'PXD020011', 28: 'PXD020011', 29: 'PXD020011', 30: 'PXD020011', 31: 'PXD020011', 32: 'PXD020011', 33: 'PXD020011', 34: 'PXD020011', 35: 'PXD020011', 36: 'PXD020011', 37: 'PXD020011', 38: 'PXD020011', 39: 'PXD020011', 40: 'PXD020011', 41: 'PXD020011', 42: 'PXD020011', 43: 'PXD020011', 44: 'PXD020011', 45: 'PXD020011', 46: 'PXD020011', 47: 'PXD020011', 48: 'PXD020011', 49: 'PXD007860', 50: 'PXD007860', 51: 'PXD007860', 52: 'PXD020011', 53: 'PXD020011', 54: 'PXD020011', 55: 'PXD020011', 56: 'MSV000096130', 57: 'MSV000096130', 58: 'MSV000096130', 59: 'MS

scan	DB_search_score	precursor_score	DB_proteins	precursor_proteins	precursor_better	charge	peptide	peptide_demod	peptide_length	reason_mismatch	MinNTermAdd	minNTermSubtract	MinCTermAdd	minCTermSubtract
6352	-25.036301	106.825996	Q86Y38	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21
6353	-23.2418	103.260002	Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY62;A0A8I5KYH1;A0A8I5QJJ8;A0A8I5QJM4;A0A8I5QKX7	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21

In [19]:
# df_filtered = df[(df['precursor_better'] == True) & (df['precursor_proteins'].str.contains(';') == False)]

In [20]:
# df_filtered.head()

In [21]:
prec_protein = set(df['precursor_proteins'].dropna().str.split(r'[;-]').str[0])
# print(f"Number of unique precursor proteins: {len(prec_protein)}")
print(f"Number of unique precursor proteins: {len(prec_protein)}")

db_protein = set(df2['precursor_proteins'].dropna().str.split(r'[;-]').str[0])
print(f"Number of unique DB proteins: {len(db_protein)}")

Number of unique precursor proteins: 331
Number of unique DB proteins: 454


In [25]:
import re

# Create a dictionary to store the data for each precursor protein
#iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

protein_data = {}

for idx, row in df.iterrows():
    # Create a dictionary to store the data for each precursor protein
    #iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

    proteins = row['precursor_proteins']
    if pd.isna(proteins):
        continue
    protein = re.split(r'[;-]', proteins)[0]
    peptide = row['peptide_demod']
    if len(peptide) <9:
        continue
    scan = row['scan']
    dataset = scan_to_dataset.get(scan, 'Unknown')
    reason_mismatch = row['reason_mismatch']
    evidence = False
    source = 'Missed'

    
    if protein not in protein_data:
        protein_data[protein] = {
            'precursor_protein': protein,
            'precursor_id_number': 0,
            'evidence_peptides_scans': []
        }
    
    protein_data[protein]['precursor_id_number'] += 1
    protein_data[protein]['evidence_peptides_scans'].append((peptide,dataset, scan,reason_mismatch,evidence,source))

for protein, data in protein_data.items():
    # print(data)
    peptides_temp = []
    peptides_temp_atomic = []
    peptides_atomic_max_l_dict = {}
    for peptide, dataset, _, _, _,_ in protein_data[protein]['evidence_peptides_scans']:
        peptides_temp.append((peptide,dataset))

    # initialize peptide dict
    for p_d in peptides_temp:
        peptide = p_d[0]  # Extract the peptide sequence from the tuple
        dataset = p_d[1]  # Extract the dataset from the tuple
        peptides_atomic_max_l_dict[p_d] = len(peptide)  # Initialize the dictionary with the length of each peptide

    peptides_to_keep = set(peptides_temp)  # Create a set to track peptides to keep
    for pd1 in list(peptides_temp):  # Iterate over the peptides in peptides_temp
        peptide1 = pd1[0]  # Extract the peptide sequence from the tuple
        dataset1 = pd1[1]  # Extract the dataset from the tuple
        if len(peptide1) < 9:
            peptides_to_keep.discard(pd1)  # Mark p1 for removal if its length is less than 9
            continue
        for pd2 in list(peptides_temp):  # Compare p1 with other peptides in peptides_temp
            peptide2 = pd2[0]  # Extract the peptide sequence from the tuple
            dataset2 = pd2[1]  # Extract the dataset from the tuple

            if peptide1 in peptide2 and peptide1 != peptide2 and dataset1 == dataset2:
                # Check if p1 is contained within p2 and is not the same as p2
                if peptides_atomic_max_l_dict[pd1] < len(peptide2):
                    # print(f"Peptide {p1} is contained in {p2}")
                    peptides_atomic_max_l_dict[pd1] = len(peptide2)

                    if pd1 not in peptides_temp_atomic:
                        peptides_temp_atomic.append(pd1)
                    peptides_to_keep.discard(pd1)  # Mark p1 for removal if it is smaller and overlaps with p2
                continue

    peptides_temp = list(peptides_to_keep)  # Update peptides_temp with the peptides to keep
    peptides_temp = list(set(peptides_temp))
    
    # print(peptides_temp)
 
    data['n_noncontained_le9'] = len(peptides_temp)
#peptide,dataset, scan,reason_mismatch,evidence,source
    for peptide, dataset, _, _, evidence, _ in protein_data[protein]['evidence_peptides_scans']:
        if (peptide, dataset) in peptides_temp:
            for i, (pep, ds, scan, reason, _, _) in enumerate(protein_data[protein]['evidence_peptides_scans']):
                if (pep, ds) == (peptide, dataset):
                    protein_data[protein]['evidence_peptides_scans'][i] = (pep, ds, scan, reason, True, 'Missed')
# Create the new dataframe
df_protein_summary = pd.DataFrame(protein_data.values())

In [26]:
df_protein_summary.to_csv('mismatched_protein_level.tsv', sep='\t', index=False)

In [27]:
matched_protein_data = {}

for idx, row in df2.iterrows():
    proteins = row['precursor_proteins']
    if pd.isna(proteins):
        continue

    protein = re.split(r'[;-]', proteins)[0]
    peptide = row['peptide_demod']
    if len(peptide) < 9:
        continue

    scan = row['scan']
    dataset = scan_to_dataset.get(scan, 'Unknown')
    reason_mismatch = row['reason_mismatch']
    evidence = False
    source = 'Matched'

    if protein not in matched_protein_data:
        matched_protein_data[protein] = {
            'precursor_protein': protein,
            'precursor_id_number': 0,
            'evidence_peptides_scans': []
        }

    matched_protein_data[protein]['precursor_id_number'] += 1
    matched_protein_data[protein]['evidence_peptides_scans'].append(
        (peptide, dataset, scan, reason_mismatch, evidence, source)
    )

for protein, data in matched_protein_data.items():
    peptides_temp = []
    peptides_temp_atomic = []
    peptides_atomic_max_l_dict = {}

    for peptide, dataset, _, _, _, _ in data['evidence_peptides_scans']:
        peptides_temp.append((peptide, dataset))

    for p_d in peptides_temp:
        peptides_atomic_max_l_dict[p_d] = len(p_d[0])

    peptides_to_keep = set(peptides_temp)
    for pd1 in list(peptides_temp):
        peptide1, dataset1 = pd1
        if len(peptide1) < 9:
            peptides_to_keep.discard(pd1)
            continue

        for pd2 in list(peptides_temp):
            peptide2, dataset2 = pd2
            if peptide1 in peptide2 and peptide1 != peptide2 and dataset1 == dataset2:
                if peptides_atomic_max_l_dict[pd1] < len(peptide2):
                    peptides_atomic_max_l_dict[pd1] = len(peptide2)
                    if pd1 not in peptides_temp_atomic:
                        peptides_temp_atomic.append(pd1)
                    peptides_to_keep.discard(pd1)

    peptides_temp = list(set(peptides_to_keep))
    data['n_noncontained_le9'] = len(peptides_temp)

    for peptide, dataset, _, _, _, _ in data['evidence_peptides_scans']:
        if (peptide, dataset) in peptides_temp:
            for i, (pep, ds, scan, reason, _, _) in enumerate(data['evidence_peptides_scans']):
                if (pep, ds) == (peptide, dataset):
                    data['evidence_peptides_scans'][i] = (pep, ds, scan, reason, True, 'Matched')

df_matched_protein_summary = pd.DataFrame(matched_protein_data.values())
df_matched_protein_summary.to_csv('matched_protein_level.tsv', sep='\t', index=False)


In [28]:
mismatch_summary = df_protein_summary.rename(columns={
    'precursor_id_number': 'mismatch_precursor_id_number',
    'evidence_peptides_scans': 'mismatch_evidence_peptides_scans',
    'n_noncontained_le9': 'mismatch_n_noncontained_le9'
})

matched_summary = df_matched_protein_summary.rename(columns={
    'precursor_id_number': 'matched_precursor_id_number',
    'evidence_peptides_scans': 'matched_evidence_peptides_scans',
    'n_noncontained_le9': 'matched_n_noncontained_le9'
})

combined_protein_summary = pd.merge(
    mismatch_summary,
    matched_summary,
    on='precursor_protein',
    how='outer'
)

def _ensure_list(x):
    return x if isinstance(x, list) else []

combined_protein_summary['mismatch_evidence_peptides_scans'] = \
    combined_protein_summary['mismatch_evidence_peptides_scans'].apply(_ensure_list)
combined_protein_summary['matched_evidence_peptides_scans'] = \
    combined_protein_summary['matched_evidence_peptides_scans'].apply(_ensure_list)

combined_protein_summary['evidence_peptides_scans'] = (
    combined_protein_summary['mismatch_evidence_peptides_scans']
    + combined_protein_summary['matched_evidence_peptides_scans']
)

combined_protein_summary.to_csv('combined_protein_level.tsv', sep='\t', index=False)


In [29]:
def _atomic_noncontained(peptide_records):
    peptide_dataset_pairs = []

    for rec in peptide_records:
        if not isinstance(rec, (tuple, list)) or len(rec) < 2:
            continue

        # Expected format: (peptide, dataset, scan, reason, evidence, source)
        if len(rec) >= 6:
            peptide, dataset = rec[0], rec[1]
        # Backward-compatibility for old format: (peptide, scan, reason, evidence, source)
        elif len(rec) == 5:
            peptide, scan = rec[0], rec[1]
            dataset = scan_to_dataset.get(scan, 'Unknown')
        else:
            continue

        if isinstance(peptide, str) and len(peptide) >= 9:
            peptide_dataset_pairs.append((peptide, dataset))

    peptides_to_keep = set(peptide_dataset_pairs)

    for p1, d1 in peptide_dataset_pairs:
        for p2, d2 in peptide_dataset_pairs:
            # Only compare within the same dataset
            if d1 != d2:
                continue
            if p1 != p2 and p1 in p2 and len(p1) < len(p2):
                peptides_to_keep.discard((p1, d1))
                break

    return peptides_to_keep


combined_n_noncontained = []
updated_evidence_records = []

for _, row in combined_protein_summary.iterrows():
    records = row['evidence_peptides_scans']
    if not isinstance(records, list):
        records = []

    atomic_pairs = _atomic_noncontained(records)
    combined_n_noncontained.append(len(atomic_pairs))

    updated_records = []
    for rec in records:
        if not isinstance(rec, (tuple, list)):
            continue

        if len(rec) >= 6:
            pep, dataset, scan, reason, _, source = rec[:6]
        elif len(rec) == 5:
            pep, scan, reason, _, source = rec
            dataset = scan_to_dataset.get(scan, 'Unknown')
        else:
            continue

        evidence = (pep, dataset) in atomic_pairs
        updated_records.append((pep, dataset, scan, reason, evidence, source))

    updated_evidence_records.append(updated_records)

combined_protein_summary['combined_n_noncontained_le9'] = combined_n_noncontained
combined_protein_summary['evidence_peptides_scans'] = pd.Series(
    updated_evidence_records,
    index=combined_protein_summary.index,
    dtype='object'
)


combined_protein_summary.to_csv('combined_protein_level.tsv', sep='\t', index=False)